# RoPE（旋转位置编码）：从绝对位置到相对位置

**学习目标：**先理解为什么需要重新设计位置信息进入 Attention 的方式，再理解二维旋转如何工作，最后把它推广到多头 Attention。这里不实现完整 `nn.Module`，完整模型单独放在 `.py` 中。

## 一、从绝对位置编码开始：它究竟怎样进入 Attention？

最初的 Transformer 将位置编码与输入相加，然后再计算 Q、K、V：

$$
 h_i=x_i+PE(i),\qquad q_i=W_Qh_i,\qquad k_j=W_Kh_j.
$$

这里将单个 Token 的特征向量按列向量书写；代码中仍然将多个 Token 按行存储。

因此，**Q、K 原本就可以包含绝对位置信息**，并不是没有位置编码。问题也不是“绝对位置编码无效”，而是我们还可以问：对于某些 Token 之间的关系，具体处于第几个位置真的很重要吗？

例如，短语“很 开心”可以出现在位置 `(1,2)`，也可以出现在位置 `(101,102)`。虽然绝对位置发生变化，但“很”与“开心”相邻这一关系没有变化。如果我们关心的是**相对位置关系**，就希望位置处理方式能够直接体现这种不随整体平移而改变的关系。

注意：这是一种有用的**设计偏好**，不是说语言中只有相对位置重要，也不是要求所有 Attention 分数都仅由距离决定。内容匹配仍然由 Q、K 决定。

## 二、回顾 Sinusoidal PE：为什么每两个特征维度使用一组 sin 和 cos？

原始 Transformer 的正余弦绝对位置编码为：

$$
PE(pos,2t)=\sin\left(\frac{pos}{10000^{2t/d_{model}}}\right),\qquad
PE(pos,2t+1)=\cos\left(\frac{pos}{10000^{2t/d_{model}}}\right).
$$

其中 $pos$ 是 Token 在序列中的位置编号，$d_{model}$ 是每个 Token 的特征维度，$t$ 是特征分组的编号。令 $\theta_t=10000^{-2t/d_{model}}$，则同一组的两个特征维度使用相同的频率 $\theta_t$。对于长度为 $L$ 的序列，位置编码矩阵的形状为 $(L,d_{model})$，每一行对应一个 Token。

### 1. 为什么不让每个特征维度都使用不同频率的 sin？

假设 $d_{model}=4$，每个 Token 都需要一个四维的位置编码向量。可以考虑以下两种方案：

| | 维度 0 | 维度 1 | 维度 2 | 维度 3 |
|---|---|---|---|---|
| 全部使用 sin | $\sin(pos\theta_0)$ | $\sin(pos\theta_1)$ | $\sin(pos\theta_2)$ | $\sin(pos\theta_3)$ |
| 原始 Transformer | $\sin(pos\theta_0)$ | $\cos(pos\theta_0)$ | $\sin(pos\theta_1)$ | $\cos(pos\theta_1)$ |

第一种方案有四种频率，第二种只有两种频率，但每种频率同时使用 sin 和 cos。两种方案都没有改变 $d_{model}$，两两分组的是**特征维度，而不是 Token**。

只使用 sin 也可以构造位置编码。那么，为什么原始 Transformer 要让两个特征维度共享一个频率呢？

### 2. 从位置变化出发，理解为什么需要 sin 和 cos 配对

假设当前 Token 位于 $pos$，另一个 Token 位于 $pos+r$，其中 $r$ 表示有符号的相对位置偏移。对于某一个固定频率 $\theta$，如果只使用 sin，两个位置的编码分别为 $\sin(pos\theta)$ 和 $\sin((pos+r)\theta)$。

利用三角函数的和角公式：

$$
\sin((pos+r)\theta)=\sin(pos\theta)\cos(r\theta)+\cos(pos\theta)\sin(r\theta).
$$

可以发现，**如果希望根据原来的位置编码和相对偏移 $r$ 直接计算出新位置的编码，不仅需要原来的 sin，还需要相同频率的 cos。** 其他频率的 sin 并不能直接代替这一项。

因此，可以将同一个频率的 sin 和 cos 一起保存在位置编码中。取完整位置编码向量的第 $t$ 组（特征维度 $2t$、$2t+1$）：

$$
PE_t(pos)=\begin{bmatrix}\sin(pos\theta_t)\\\cos(pos\theta_t)\end{bmatrix}.
$$

当位置从 $pos$ 移动到 $pos+r$ 时，两个特征维度分别满足：

$$
\begin{aligned}
\sin((pos+r)\theta_t)&=\sin(pos\theta_t)\cos(r\theta_t)+\cos(pos\theta_t)\sin(r\theta_t),\\
\cos((pos+r)\theta_t)&=\cos(pos\theta_t)\cos(r\theta_t)-\sin(pos\theta_t)\sin(r\theta_t).
\end{aligned}
$$

将这两个公式写成矩阵形式：

$$
\underbrace{\begin{bmatrix}\sin((pos+r)\theta_t)\\\cos((pos+r)\theta_t)\end{bmatrix}}_{PE_t(pos+r)}
=
\underbrace{\begin{bmatrix}\cos(r\theta_t)&\sin(r\theta_t)\\-\sin(r\theta_t)&\cos(r\theta_t)\end{bmatrix}}_{\text{仅由相对偏移决定}}
\underbrace{\begin{bmatrix}\sin(pos\theta_t)\\\cos(pos\theta_t)\end{bmatrix}}_{PE_t(pos)}.
$$

例如，从位置 1 移动到 3、从位置 10 移动到 12，或者从位置 100 移动到 102，偏移 $r$ 都等于 2，因此对同一频率使用的变换矩阵完全相同。

这就是 sin 和 cos 配对的关键优势：**对于固定的相对偏移，可以使用同一个线性变换得到新位置的编码，而不需要针对不同的绝对位置设计不同的变换。** 只用 sin 并不意味着模型无法学习相对位置，但一般不具备这种简单的逐组线性变换性质。

不同频率的作用则是提供不同尺度的位置变化：高频变化快，低频变化慢。每组用两个维度描述同一频率的完整正余弦信息，多组频率共同组成 $d_{model}$ 维位置向量。

### 3. 正余弦位置编码与 Attention 中的相对位置

现在只看这组位置向量之间的点积：

$$
\begin{aligned}
PE_t(i)^\top PE_t(j)
&=\sin(i\theta_t)\sin(j\theta_t)+\cos(i\theta_t)\cos(j\theta_t)\\
&=\cos((i-j)\theta_t).
\end{aligned}
$$

**单独比较两个正余弦位置向量，点积只与相对距离有关。** 但原始 Transformer 并不是单独比较两条位置向量，而是先将位置编码与输入内容相加，再计算 Q、K。

为了看清这种相加的影响，暂时省略 Q、K 的线性投影，展开：

$$
\begin{aligned}
(x_i+PE(i))^\top(x_j+PE(j))
={}&\underbrace{x_i^\top x_j}_{\text{内容与内容}}\\
&+\underbrace{x_i^\top PE(j)+PE(i)^\top x_j}_{\text{内容与位置的交叉项}}\\
&+\underbrace{PE(i)^\top PE(j)}_{\text{位置与位置}}.
\end{aligned}
$$

最后一项能够体现相对距离，但两个交叉项仍可能受到绝对位置 $i$、$j$ 的影响。真实模型还会经过 $W_Q,W_K$ 投影，因此不能仅凭正余弦位置向量的点积性质，就认为**整个 Attention 分数**必然只与相对距离有关。

此前 Shaw、Transformer-XL、T5 和 ALiBi 都采用不同方式，让相对位置更直接地参与 Attention。这里还有一个值得利用的结果：前面推导的线性变换矩阵本身就包含相对位置信息。接下来研究能否将这个矩阵用于 Attention 的内容匹配。

## 三、从加法式位置编码到旋转位置编码

回顾此前学习的位置编码，它们引入位置信息的方式有所不同。

**Sinusoidal Position Encoding：** 根据 Token 的绝对位置计算正余弦位置向量，并直接加到输入特征上：

$$h_i=x_i+PE(i).$$

随后使用包含位置信息的 $h_i$ 生成 Q、K、V。因此，位置信息在进入 Attention 之前就已经与内容信息进行了融合。

**T5、Transformer-XL 和 ALiBi：** 不再简单地将位置向量加到输入特征上，而是将相对位置信息引入 Attention Score 的计算。例如，T5 和 ALiBi 可以用常见的缩放点积形式表示为：

$$
\ell_{ij}=\underbrace{\frac{q_i^\top k_j}{\sqrt{d_k}}}_{\text{内容匹配分数}}+\underbrace{b(i-j)}_{\text{相对位置偏置}}.
$$

Transformer-XL 采用更复杂的内容与位置交互项，同样通过附加打分项引入相对位置，而不只是增加一个与内容无关的标量偏置。

那么，除了在输入特征或匹配分数上做加法，**能否通过改变 Q、K 向量本身来引入位置信息？**

### 1. 为什么可以考虑旋转？

回忆普通 Attention 中的匹配分数，暂时省略缩放系数：

$$s_{ij}=q_i^\top k_j.$$

其中 $s_{ij}$ 就是 $QK^\top$ 矩阵的第 $i$ 行、第 $j$ 列。从几何角度来看，两个非零向量的点积为：

$$q_i^\top k_j=\|q_i\|\|k_j\|\cos\phi.$$

其中 $\phi$ 是两个向量之间的夹角。因此，即使不改变向量的长度，只改变向量之间的夹角，也能够影响匹配分数。

例如，两个单位向量的夹角从 $30^\circ$ 变成 $90^\circ$，点积就从 $\cos30^\circ\approx0.866$ 变为 $\cos90^\circ=0$，而向量长度保持不变。

旋转恰好能够在保持向量长度的同时改变方向。因此，可以考虑通过旋转 Q、K 向量的方式引入位置信息。

### 2. 为什么旋转后的点积包含相对位置信息？

假设 Q、K 尚未显式引入位置信息。如果保持两个内容向量不变，只改变它们在序列中的位置，那么普通点积 $q_i^\top k_j$ 不会因此改变。

T5 和 ALiBi 的做法是在分数上加入位置偏置，使相对距离能够影响最终的匹配分数。现在换一种思路：**不直接改变分数，而是让相对距离决定参与点积的向量方向。**

回顾上一节通过正余弦和角公式得到的矩阵（暂时只考虑一个频率组）：

$$
M(r)=\begin{bmatrix}
\cos(r\theta)&\sin(r\theta)\\
-\sin(r\theta)&\cos(r\theta)
\end{bmatrix}.
$$

这是一个由相对位移 $r$ 决定的二维旋转矩阵，按逆时针为正的约定，旋转角度为 $-r\theta$。因此，可以将 Query 与 Key 的相对距离 $D_{ij}=i-j$ 作为矩阵的输入，尝试在计算点积之前，先对二维 Key 向量进行旋转：

$$k_j^{(i)}=M(D_{ij})k_j.$$

再将旋转后的 Key 与原来的 Query 计算点积：

$$
s_{ij}=q_i^\top k_j^{(i)}=q_i^\top M(D_{ij})k_j.
$$

这里的 $k_j^{(i)}$ 表示根据当前 Query 位置变换后的 Key。这个公式是一种新的匹配设计，并不是普通点积的恒等变形。

**为什么这样就引入了相对位置信息？** 假设原始内容向量保持不变：

$$
q=k=\begin{bmatrix}1\\0\end{bmatrix},\qquad \theta=30^\circ.
$$

在普通 Attention 中，无论两个 Token 相距多远，它们的点积始终为 $q^\top k=1$。引入相对旋转后：

| 相对距离 | Key 的顺时针旋转角度 | 普通点积 | 旋转后的点积 |
|---|---:|---:|---:|
| $D=0$ | $0^\circ$ | $1$ | $1$ |
| $D=1$ | $30^\circ$ | $1$ | $\cos30^\circ\approx0.866$ |
| $D=2$ | $60^\circ$ | $1$ | $\cos60^\circ=0.5$ |

虽然原始内容向量没有变化，但相对距离改变了 Key 的旋转角度，进而改变 Q、K 的夹角和点积。这就是通过旋转引入相对位置信息的原因。

| 方式 | 匹配分数 | 位置信息的引入方式 |
|---|---|---|
| 普通 Attention（未引入位置） | $q_i^\top k_j/\sqrt{d_k}$ | 仅计算内容匹配 |
| T5、ALiBi（缩放点积示意） | $q_i^\top k_j/\sqrt{d_k}+b(D_{ij})$ | 在分数上加入位置偏置 |
| 相对位置旋转 | $q_i^\top M(D_{ij})k_j/\sqrt{d_k}$ | 通过旋转改变参与点积的向量 |

**相对距离变化 → 旋转角度变化 → 向量之间的夹角可能变化 → 匹配分数可能变化。**

上面的例子并不意味着距离越远分数一定越小。实际分数由原始内容向量和相对旋转角度共同决定。

### 3. 接下来需要解决什么问题？

直接采用 $q_i^\top M(i-j)k_j$ 时，旋转角度同时依赖 Query 和 Key 的位置。因此，同一个 Key 面对不同 Query 时，需要进行不同的向量旋转。下一节研究如何在保持匹配分数不变的前提下，减少这些额外的位置变换。


## 四、为什么 RoPE 最终分别旋转 Q、K？

### 1. 先统一旋转矩阵的表示

按照常见约定，$R(\phi)$ 表示逆时针旋转 $\phi$：

$$
R(\phi)=\begin{bmatrix}\cos\phi&-\sin\phi\\\sin\phi&\cos\phi\end{bmatrix},\qquad
R(\phi)\begin{bmatrix}a\\b\end{bmatrix}=
\begin{bmatrix}a\cos\phi-b\sin\phi\\a\sin\phi+b\cos\phi\end{bmatrix}.
$$

前面 Sinusoidal PE 按照先 sin、后 cos 的特征顺序排列，所以从和角公式中得到的是顺时针旋转矩阵 $M(r)=R(-r\theta)$。因此沿用 $D_{ij}=i-j$ 时：

$$M(D_{ij})=R(-D_{ij}\theta)=R((j-i)\theta).$$

这是同一个矩阵的两种写法，并没有改变距离定义。原始 PE 中的 sin、cos 是位置向量的两个特征；这里旋转矩阵中的 sin、cos 是变换系数。

### 2. 为什么需要拆分相对位置旋转？

上一节已经得到了一种能够表达相对位置的匹配方式（暂时不考虑缩放）：

$$s_{ij}=q_i^\top R((j-i)\theta)k_j.$$

既然这一形式已经能够体现相对位置，为什么还要改变计算方式？下面以位置 0、1、2 的三个 Token 为例，暂时不加因果 Mask。

在普通双向 Self-Attention 中，每个 Key 都需要与 $L$ 个 Query 计算点积，但参与计算的 Key 向量始终不变。引入相对旋转后，以位置 1 的 Key 为例：

$$
\begin{aligned}
s_{01}&=q_0^\top M(-1)k_1\\
s_{11}&=q_1^\top M(0)k_1\\
s_{21}&=q_2^\top M(1)k_1
\end{aligned}
$$

虽然相对距离及其对应的旋转矩阵 $M(-1)$、$M(0)$、$M(1)$ 可以提前计算，但这些矩阵仍然需要分别与实际的 $k_1$ 相乘，得到对应的旋转结果。这里的额外开销并不是计算 $i-j$ 的减法，而是旋转矩阵对内容向量的变换。

对于长度为 $L$ 的序列，每个 Key 都需要考虑与 $L$ 个 Query 的相对旋转。直接逐对实现时，除了普通 Attention 原有的 $L^2$ 次点积，还要考虑 $L^2$ 次向量旋转（包含可以跳过的零角度旋转）。

**那么，能否在计算点积之前，分别根据 Query 和 Key 自身的绝对位置完成旋转，使旋转后的向量能够在后续的匹配过程中重复使用？**

如果能够做到这一点，并且保证最终的匹配分数与直接相对旋转完全相同，就只需要对 $L$ 个 Query 和 $L$ 个 Key 分别进行一次旋转，共 $2L$ 次向量变换。这样可以将额外的位置变换开销从随 $L^2$ 增长降为随 $L$ 增长，而原有的点积计算仍然保留。

接下来利用旋转矩阵的数学性质，研究如何将同时依赖两个位置的旋转拆成两个独立的位置变换。

### 3. 尝试将相对旋转拆成两个独立部分

观察相对旋转角度：

$$(j-i)\theta=-i\theta+j\theta.$$

其中 $-i\theta$ 只与 Query 的位置有关，$j\theta$ 只与 Key 的位置有关。能不能像拆分角度一样，把旋转矩阵也拆开？

假设一个向量先逆时针旋转 $30^\circ$，再逆时针旋转 $60^\circ$，最终相当于直接旋转 $90^\circ$。旋转矩阵满足：

$$R(a)R(b)=R(a+b).$$

因此：

$$R((j-i)\theta)=R(-i\theta)R(j\theta).$$

将这个结果代回匹配分数，并利用矩阵乘法的结合律进行分组（没有交换矩阵的顺序）：

$$
\begin{aligned}
s_{ij}&=q_i^\top R((j-i)\theta)k_j\\
&=\underbrace{q_i^\top R(-i\theta)}_{\text{只与 Query 有关}}
\underbrace{R(j\theta)k_j}_{\text{只与 Key 有关}}.
\end{aligned}
$$

原来的旋转矩阵同时依赖 Query 和 Key 的位置，现在已经拆成两个可以独立计算的部分。这样就有机会在计算点积之前，分别完成 Q、K 的位置变换。

### 4. 如何将这两个部分分别写成新的 Query 和 Key？

在这一节的数学推导中，单个 $q_i,k_j$ 都按列向量表示，形状为 $(2,1)$；计算点积时才将 Query 转置为行向量。

**先看 Key 部分。** $R(j\theta)$ 的形状为 $(2,2)$，与 $k_j$ 相乘后仍然得到一个二维列向量。因此可以直接定义：

$$\widetilde k_j=R(j\theta)k_j.$$

**再看 Query 部分。** $q_i^\top R(-i\theta)$ 的形状是 $(1,2)$，它已经是一个行向量。为了与普通 Attention 的点积形式一致，需要将它整理成某个新 Query 列向量的转置形式，即 $\widetilde q_i^\top$。

旋转矩阵是正交矩阵，转置等于逆矩阵。反向旋转可以抵消原来的旋转：

$$R(-a)R(a)=I,\qquad R(-a)=R(a)^\top.$$

因此：

$$q_i^\top R(-i\theta)=q_i^\top R(i\theta)^\top.$$

再利用矩阵乘积的转置规则 $(AB)^\top=B^\top A^\top$：

$$q_i^\top R(i\theta)^\top=(R(i\theta)q_i)^\top.$$

括号内部就是一个经过旋转的 Query 列向量，因此定义：

$$\widetilde q_i=R(i\theta)q_i.$$

现在两个独立的位置变换都已经得到：

$$\boxed{\begin{aligned}
\widetilde q_i&=R(i\theta)q_i,\\
\widetilde k_j&=R(j\theta)k_j.
\end{aligned}}$$

注意：拆分时出现的 $R(-i\theta)$ 位于 Query 的行向量一侧，将它整理成列向量形式后，Query 实际使用的旋转角度仍然是正的 $i\theta$，与 Key 使用相同的旋转方向约定。

### 5. 拆分之后，为什么就可以直接使用普通矩阵乘法？

重新观察前面的三个 Token。对于位置 1 的 Key，只需提前计算一次：

$$\widetilde k_1=R(\theta)k_1.$$

三个 Query 也只需分别提前计算一次：

$$\begin{aligned}
\widetilde q_0&=R(0)q_0,\\
\widetilde q_1&=R(\theta)q_1,\\
\widetilde q_2&=R(2\theta)q_2.
\end{aligned}$$

| 匹配对象 | 直接进行相对旋转 | 拆分后的点积 |
|---|---|---|
| Query 0 | $q_0^\top R(\theta)k_1$ | $\widetilde q_0^\top\widetilde k_1$ |
| Query 1 | $q_1^\top k_1$ | $\widetilde q_1^\top\widetilde k_1$ |
| Query 2 | $q_2^\top R(-\theta)k_1$ | $\widetilde q_2^\top\widetilde k_1$ |

每一行中左右两种计算的结果完全相同，但右边三行都复用了同一个 $\widetilde k_1$。其他 Key 也可以按照同样的方式提前旋转。每个旋转后的 Query 同样可以与所有 Key 匹配。

对于长度为 $L$ 的双向 Self-Attention，以每次变换一个完整向量为单位：直接按匹配对进行旋转需要考虑 $L^2$ 次变换（零角度可跳过），而拆分后只需要 $L$ 次 Query 变换和 $L$ 次 Key 变换，共 $2L$ 次。这里比较的是逐对直接实现与拆分实现的位置变换次数，不是整个 Attention 的计算复杂度；所有 Query–Key 点积仍需计算。

将各个旋转后的列向量转置并按行堆叠，就得到普通的 Query 和 Key 矩阵（本二维示例中形状为 $(L,2)$）：

$$
\widetilde Q=\begin{bmatrix}\widetilde q_0^\top\\\widetilde q_1^\top\\\widetilde q_2^\top\end{bmatrix},\qquad
\widetilde K=\begin{bmatrix}\widetilde k_0^\top\\\widetilde k_1^\top\\\widetilde k_2^\top\end{bmatrix}.
$$

于是可以直接计算整个匹配分数矩阵：

$$S=\widetilde Q\widetilde K^\top\quad (L,L).$$

其中每一个元素仍然满足：

$$\boxed{\widetilde q_i^\top\widetilde k_j=q_i^\top R((j-i)\theta)k_j=q_i^\top R(-D_{ij}\theta)k_j.}$$

**这就是 RoPE 的计算形式：分别按照各自的绝对位置旋转 Q、K，却能在点积过程中体现相对位置关系。** 不需要显式构造每对 Query–Key 的旋转矩阵，就能保留标准 Attention 的矩阵乘法形式。

### 6. 用具体角度理解这种等价关系

仍取原始 $q=k=(1,0)^\top$，固定 $\theta=30^\circ$：

| | 第一组 | 第二组 |
|---|---:|---:|
| Query 位置 $i$ | 2 | 4 |
| Key 位置 $j$ | 1 | 3 |
| Q 逆时针旋转 | $60^\circ$ | $120^\circ$ |
| K 逆时针旋转 | $30^\circ$ | $90^\circ$ |
| 两个向量的夹角 | $30^\circ$ | $30^\circ$ |
| 旋转后点积 | $0.866$ | $0.866$ |

第二组中两个 Token 都向后移动了两个位置，因此 Q、K 都额外旋转了 $60^\circ$。两个向量共同旋转相同角度不会改变长度和相互夹角，所以点积不变。对于任意固定的原始 Q、K，这个性质同样成立，而不要求它们本来方向相同。

如果角度改成 $pos^2\theta$，两个位置对应的角度差会变成 $(j-i)(j+i)\theta$，不再只由相对距离决定。这说明使用 $pos\theta$ 的意义是让角度差只与相对偏移有关。

实际模型中，不同 Token 的内容向量可能不同，因此相同距离并不保证相同分数；这里只说明 RoPE 引入的显式位置变换具有相对位置性质。下面用代码验证两组位置的旋转后点积。

In [1]:
import torch
import math

# 这里只验证二维 RoPE 的核心性质，不重复完整 Q、K、V 投影
# 与之前 Sinusoidal PE 的 (sin,cos) 位置向量不同，这里的 q、k 是任意内容向量
q = torch.tensor([1.0, 0.0])
k = torch.tensor([1.0, 0.0])
theta = math.pi / 6  # 用30°方便观察，不是实际 RoPE 的默认频率


def rotate_2d(x, position):
    # 采用常见的逆时针旋转矩阵 R(+position*theta)，不等于直接构造 Sinusoidal PE
    angle = position * theta
    c, s = math.cos(angle), math.sin(angle)
    return torch.stack((x[0] * c - x[1] * s, x[0] * s + x[1] * c))


# 位置(2,1)与位置(4,3)虽然绝对位置不同，但相对距离相同
score_21 = torch.dot(rotate_2d(q, 2), rotate_2d(k, 1))
score_43 = torch.dot(rotate_2d(q, 4), rotate_2d(k, 3))

print(f"位置 (2,1) 的旋转后点积：{score_21.item():.3f}")
print(f"位置 (4,3) 的旋转后点积：{score_43.item():.3f}")
assert torch.allclose(score_21, score_43, atol=1e-6)

# 再检查拆分是否真的等价于逐对相对旋转：这里使用不同的原始内容向量
Q_demo = torch.tensor([[1., 2.], [2., -1.], [0.5, 3.]])  # (3,2)，每行一个 Query
K_demo = torch.tensor([[2., 1.], [-1., 2.], [3., 0.5]])  # (3,2)，每行一个 Key

# 直接计算：每个 (i,j) 都根据 j-i 重新旋转 Key
scores_pair = torch.stack([
    torch.stack([torch.dot(Q_demo[i], rotate_2d(K_demo[j], j - i)) for j in range(3)])
    for i in range(3)
])  # (3,3)

# RoPE：每个 Query、Key 只旋转一次，再直接矩阵相乘
Q_rot = torch.stack([rotate_2d(Q_demo[i], i) for i in range(3)])  # (3,2)
K_rot = torch.stack([rotate_2d(K_demo[j], j) for j in range(3)])  # (3,2)
scores_rope = Q_rot @ K_rot.T  # (3,3)
print("逐对相对旋转与 RoPE 的全部分数是否一致：", torch.allclose(scores_pair, scores_rope, atol=1e-6))
assert torch.allclose(scores_pair, scores_rope, atol=1e-6)


位置 (2,1) 的旋转后点积：0.866
位置 (4,3) 的旋转后点积：0.866
逐对相对旋转与 RoPE 的全部分数是否一致： True


## 五、Q、K 实际是高维向量，二维旋转怎么办？

前面只在二维平面中旋转，但多头 Attention 中每个 Head 的维度通常是 $d_{head}=64$ 或 $128$。RoPE 的做法是：**把每个 Head 的特征维度两两分组，每组独立做二维旋转**。这里旋转的是 Q、K 的内容向量，不是把原始 Sinusoidal PE 的 `(sin,cos)` 两维直接加进去。

例如 $d_{head}=8$：

$$
(q_0,q_1)\quad(q_2,q_3)\quad(q_4,q_5)\quad(q_6,q_7).
$$

这 4 组都用同一个 Token 的绝对位置 $p$，但可以具有**不同的旋转频率**：

$$\theta_t=10000^{-2t/d_{head}},\qquad t=0,1,\ldots,d_{head}/2-1.$$

对于 $d_{head}=8$，四组频率分别为 $1,\ 0.1,\ 0.01,\ 0.001$（单位：弧度/位置）。它们在位置 $p$ 的旋转角度分别为 $p\theta_t$。这样，不同特征组以不同速度变化，不会只依赖单一角度周期。

重要的是：**每一组都仍然满足角度差与相对距离有关的性质。** 将各组的点积相加后，整个高维 Q、K 的点积也保留这一性质。

完整 Attention 中的位置是：

$$
X\rightarrow Q,K,V\rightarrow \text{拆分多头}
\rightarrow\boxed{\text{只旋转 }Q,K}
\rightarrow\frac{\widetilde Q\widetilde K^\top}{\sqrt{d_{head}}}
\rightarrow\text{Mask / Softmax / 与 V 加权求和}.
$$

- **为什么不旋转 V？** RoPE 旨在改变 Q、K 的匹配分数；V 仍提供被加权汇总的内容。
- **是否像 ALiBi 一样强制远距离扣更多分？** 不是。旋转后的分数取决于内容和相对位置，角度还具有周期性，不是简单的单调距离惩罚。
- **是否完全没有绝对位置？** 也不是。每个 Q、K 先按自己的绝对位置旋转，只是在两者的点积中，旋转部分合成为相对位置关系。

下面只写二维配对与旋转的关键步骤；完整 Q/K/V 投影和 `nn.Module` 留到单独的 `.py` 文件。

In [2]:
# 输入已经是多头拆分后的 Q 或 K：(B,H,L,d_head)，这里不重复完整 Attention
# 用相邻两维组成一对，演示如何把二维旋转推广到一个 Head 的全部特征

def apply_rope(x, positions, base=10000.0):
    B, H, L, d_head = x.shape
    if d_head % 2 != 0:
        raise ValueError("d_head 必须是偶数，才能将特征两两配对")
    if positions.numel() != L:
        raise ValueError("positions 的长度必须等于输入 Token 数量")

    # 每两个特征组成一组，共 d_head/2 组；各组使用不同的旋转频率
    t = torch.arange(d_head // 2, device=x.device, dtype=torch.float32)
    theta = base ** (-2 * t / d_head)  # (d_head/2,)

    # 角度 = Token 的绝对位置 × 当前特征组的频率
    # positions：(L,)；theta：(d_head/2,)；广播得到 (L,d_head/2)
    angles = positions.to(device=x.device, dtype=torch.float32)[:, None] * theta[None, :]
    cos = angles.cos()[None, None, :, :].to(dtype=x.dtype)  # (1,1,L,d_head/2)
    sin = angles.sin()[None, None, :, :].to(dtype=x.dtype)

    # 将最后一维按相邻两个特征分组：(B,H,L,d_head) -> (B,H,L,d_head/2,2)
    pairs = x.reshape(B, H, L, d_head // 2, 2)
    a, b = pairs.unbind(dim=-1)  # 两个张量均为 (B,H,L,d_head/2)

    # 每组分别做二维旋转：(a,b) -> (a*cos-b*sin, a*sin+b*cos)
    rotated = torch.stack((a * cos - b * sin, a * sin + b * cos), dim=-1)
    return rotated.flatten(-2)  # 恢复 (B,H,L,d_head)


# 检查：两个位置同时平移相同距离，只要原始 Q、K 不变，旋转后的 QK^T 就应保持一致
torch.manual_seed(0)
B, H, L, d_head = 1, 2, 4, 8
Q = torch.randn(B, H, L, d_head)
K = torch.randn(B, H, L, d_head)
pos = torch.arange(L)

scores_0 = apply_rope(Q, pos) @ apply_rope(K, pos).transpose(-2, -1)
scores_shift = apply_rope(Q, pos + 5) @ apply_rope(K, pos + 5).transpose(-2, -1)

print("旋转后的 Q 形状：", apply_rope(Q, pos).shape)
print("位置整体平移后，QK^T 是否保持一致：", torch.allclose(scores_0, scores_shift, atol=1e-5))
assert torch.allclose(scores_0, scores_shift, atol=1e-5)

旋转后的 Q 形状： torch.Size([1, 2, 4, 8])
位置整体平移后，QK^T 是否保持一致： True


## 六、与此前位置编码的关系

- **Sinusoidal Absolute PE：** 将正余弦位置向量加到输入上，Q、K 因而包含位置信息。
- **Shaw / Transformer-XL：** 让相对位置表示参与 Attention 的匹配计算。
- **T5 / ALiBi：** 直接在 Attention Score 上加入位置偏置。
- **RoPE：** 利用相对位置旋转可以拆分为两个绝对位置旋转的性质，在点积之前分别变换 Q、K，使位置相关的部分体现相对距离。

**一句话总结：RoPE 先按照各自的绝对位置旋转 Q、K，再利用旋转角度相减的性质，使它们的点积体现相对距离。** 它不是从 ALiBi 必然推导出来的，而是另一种位置编码设计。

> 参考：Vaswani et al., *Attention Is All You Need*, §3.5，https://arxiv.org/abs/1706.03762 。
>
> Su et al., *RoFormer: Enhanced Transformer with Rotary Position Embedding*, arXiv:2104.09864，https://arxiv.org/abs/2104.09864 。

**符号提醒：** 最初的 Sinusoidal PE 按特征维度写作 `(sin, cos)`；这里的 RoPE 是用包含 `cos`、`sin` 的矩阵旋转 Q、K，不应把两者误认为相同的向量表达式。